In [0]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

# Generate realistic baseline spending behavior: [amount, limit_ratio, geo_mismatch]
np.random.seed(101)
sample_size = 4000

amounts = np.random.exponential(scale=95.0, size=sample_size) + 12.0
ratios = np.random.beta(a=2, b=12, size=sample_size)
geo_flags = np.random.choice([0.0, 1.0], size=sample_size, p=[0.94, 0.06])

df_train = pd.DataFrame({
    "amount": amounts,
    "limit_ratio": ratios,
    "is_out_of_home": geo_flags
})

# Train Isolation Forest with ~2.5% expected contamination baseline
model = IsolationForest(
    n_estimators=150,
    max_samples="auto",
    contamination=0.025,
    random_state=42,
    n_jobs=-1
)

model.fit(df_train[["amount", "limit_ratio", "is_out_of_home"]])
print("[INFO] Baseline Isolation Forest model training complete.")

[INFO] Baseline Isolation Forest model training complete.


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS aegis_fraud_workspace.finguard.finguard_volume;

In [0]:
import os
import joblib

# Persist the model to a durable UC Volume path
artifact_path = "/Volumes/aegis_fraud_workspace/finguard/finguard_volume/artifacts/models/isolation_forest_v1.pkl"

os.makedirs(os.path.dirname(artifact_path), exist_ok=True)
joblib.dump(model, artifact_path)

print(f"[SUCCESS] Model artifact persisted to: {artifact_path}")

[SUCCESS] Model artifact persisted to: /Volumes/aegis_fraud_workspace/finguard/finguard_volume/artifacts/models/isolation_forest_v1.pkl


In [0]:
model_dir = "/Volumes/aegis_fraud_workspace/finguard/finguard_volume/artifacts/models"

display(pd.DataFrame([
    {
        "name": entry.name,
        "path": entry.path,
        "size_bytes": entry.stat().st_size
    }
    for entry in os.scandir(model_dir)
]))

name,path,size_bytes
isolation_forest_v1.pkl,/Volumes/aegis_fraud_workspace/finguard/finguard_volume/artifacts/models/isolation_forest_v1.pkl,1780057
